# Bank Account Fraud (BAF) - Preprocessing

Steps covered in this notebook:

1. **Load data** - loads `Base.csv`
2. **Cleaning** - negative missing-value codes become `NaN` with a `<col>_missing_flag` indicator, and string columns are cast to `category`
3. **Train / validation split** - months 0-6 used for training and month 7 for validation. This is time-based since fraud rate drift across months and avoids risk of data leakage
4. **Drop columns** - remove `device_fraud_count` (constant) and the two missing flags with no signal
5. **XGBoost feature set** - `source` encoded as 0/1; everything else left as-is as XGBoost handles `NaN` and categoricals natively
6. **Logistic regression feature set** - rare levels merged, add `income_top_decile`, banded `proposed_credit_limit`, `device_distinct_emails_8w` treated as categorical, then a `ColumnTransformer` that scales, log-transforms, imputes and one-hot encodes the relevant rows
7. **Check the LR preprocessor** - fit on training months only, transform both sets and confirm the output is complete and scaled
8. **Save** - write each model's train/validation features and targets to `../data/preprocessed/<model>/` as a parquet, so later notebooks can load them without rerunning this one

Anything learned from the data is fitted on the training months only and then applied to validation to avoid data leakage

Shared helpers are imported from the `baf` package in `../src/baf/` (install once with `pip install -e .` from the project root).

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from baf.config import PREPROCESSED_DIR, RAW_DATA_PATH, TARGET, TRAIN_MONTHS, VAL_MONTHS
from baf.data import MISSING_CODED_COLS, cast_categoricals, flag_missing_codes
from baf.features import COLS_TO_DROP, LR_ONEHOT_COLS, encode_source, prepare_lr_features

## Load Data

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)

(1000000, 32)


## Cleaning

Negative values in these columns are missing-value codes. They are replaced with `NaN`, and a missingness flag is kept for each since missingness carries a fraud signal as missing values are systematic

In [3]:
df = flag_missing_codes(df)

df[MISSING_CODED_COLS].isna().mean().sort_values(ascending=False).rename("missing_share")

intended_balcon_amount          0.742523
prev_address_months_count       0.712920
bank_months_count               0.253635
current_address_months_count    0.004254
session_length_in_minutes       0.002015
device_distinct_emails_8w       0.000359
Name: missing_share, dtype: float64

For consistent category handling XGBoost requires casting to `category` before the split meaning train and validation share the same category levels

In [4]:
df = cast_categoricals(df)

## Train / Validation Split

The fraud rate drifts month to month (0.87% -> 1.47%), so split is done by time rather than randomly: Months 0-6 used for training and 7 for validation. Mimics scoring future applications with past data and avoids risk of data leakage

In [5]:
train_df = df[df["month"].isin(TRAIN_MONTHS)]
val_df = df[df["month"].isin(VAL_MONTHS)]

X_train = train_df.drop(columns=["month", TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=["month", TARGET])
y_val = val_df[TARGET]

print(f"Train: {X_train.shape}, fraud rate {y_train.mean():.2%}")
print(f"Val:   {X_val.shape}, fraud rate {y_val.mean():.2%}")

Train: (903157, 36), fraud rate 1.06%
Val:   (96843, 36), fraud rate 1.47%


The validation month has a noticeably higher fraud rate (1.47%) than the training months (1.06%), so validation scores also reflect how well the model copes with drift.

## Drop Columns

As discussed in `01_eda`:

- `device_fraud_count` is a constant 0 for every row giving no information.
- `session_length_in_minutes_missing_flag` and `device_distinct_emails_8w_missing_flag` cover very few rows and their fraud rates sit at the base rate for the dataset.

In [6]:
X_train = X_train.drop(columns=COLS_TO_DROP)
X_val = X_val.drop(columns=COLS_TO_DROP)

print(X_train.shape)
print(X_val.shape)

(903157, 33)
(96843, 33)


## Feature Engineering

Two feature sets are built from the kept columns, one per model family. `source` has only two levels (INTERNET / TELEAPP), so both sets encode it as a single 0/1 column. The shared feature steps (`encode_source`, `prepare_lr_features` and the dropped columns) are in `src/baf/features.py`.

### XGBoost

XGBoost supports categorical variables natively with `enable_categorical=True`, handles `NaN` by learning a default split direction, and splits on thresholds, so no scaling, imputation or transformation is needed.

In [7]:
X_train_xgb = encode_source(X_train)
X_val_xgb = encode_source(X_val)

print(X_train_xgb.shape)
print(X_val_xgb.shape)
X_train_xgb.dtypes.value_counts()

(903157, 33)
(96843, 33)


int64       16
float64     13
category     1
category     1
category     1
category     1
Name: count, dtype: int64

The XGBoost set keeps all 33 columns: the four nominal categoricals stay as `category` (each has its own level set, so they are listed separately above), `source` is now 0/1, and the numeric columns keep their `NaN`s.

### Logistic Regression

A linear model needs more help, based on the patterns found in `01_eda`:

- **Standardise:** continuous features, `date_of_birth_distinct_emails_4w`, `customer_age`, `income`
- **Log, then standardise:** `days_since_request`, `zip_count_4w`, `bank_branch_count_8w` (one slope covers the whole range, so long tails would dominate the fit)
- **Median-fill, then standardise:** the address, bank and balcon columns; `session_length_in_minutes` also gets a log
- **One-hot, most common level as the reference:** nominal categoricals (rare levels merged), `proposed_credit_limit` bands, `device_distinct_emails_8w` levels
- **Leave as 0/1:** binaries, `source`, kept missing flags, and a new `income_top_decile` flag for the jump in the top income band

The first step below handles the row-level changes (merges, flag, banding); the `ColumnTransformer` after it handles the learned transforms.

In [8]:
# Computed on the training months only, then reused for validation
income_top_decile_cutoff = X_train["income"].quantile(0.9)
device_emails_mode = X_train["device_distinct_emails_8w"].mode()[0]

lr_train_input = prepare_lr_features(X_train, income_top_decile_cutoff, device_emails_mode)
lr_val_input = prepare_lr_features(X_val, income_top_decile_cutoff, device_emails_mode)

- Rare levels (`payment_type` AE, `employment_status` CG, `housing_status` BF/BG, each under 0.2% of rows) are merged into the level with the closest training fraud rate, so the one-hot coefficients aren't overfitted on a few hundred rows.
- `proposed_credit_limit` is banded because fraud sits around 0.7-2% up to 1,500 and then jumps to 12-17% above it.
- `income_top_decile_cutoff` and `device_emails_mode` are computed from `X_train` only and reused for validation.
- The rare-level merges, credit-limit bands and these steps are defined in `prepare_lr_features` (`src/baf/features.py`).

In [ ]:
lr_scale_cols = [
    "credit_risk_score", "name_email_similarity", "velocity_6h", "velocity_24h", "velocity_4w",
    "date_of_birth_distinct_emails_4w",  # skew ~0.7, mild enough to skip the log
    "customer_age", "income",
]
lr_log_scale_cols = ["days_since_request", "zip_count_4w", "bank_branch_count_8w"]
lr_impute_scale_cols = [
    "prev_address_months_count", "current_address_months_count", "bank_months_count", "intended_balcon_amount",
]
lr_impute_log_scale_cols = ["session_length_in_minutes"]
lr_passthrough_cols = [
    "email_is_free", "phone_home_valid", "phone_mobile_valid", "has_other_cards",
    "foreign_request", "keep_alive_session", "source", "income_top_decile",
    "prev_address_months_count_missing_flag", "bank_months_count_missing_flag",
    "intended_balcon_amount_missing_flag", "current_address_months_count_missing_flag",
]

lr_all_cols = (lr_scale_cols + lr_log_scale_cols + lr_impute_scale_cols + lr_impute_log_scale_cols
               + LR_ONEHOT_COLS + lr_passthrough_cols)
assert sorted(lr_all_cols) == sorted(lr_train_input.columns), set(lr_all_cols) ^ set(lr_train_input.columns)

# Each one-hot feature's most common training level is dropped and becomes the reference
lr_reference_levels = []
for col in LR_ONEHOT_COLS:
    most_common_level = lr_train_input[col].mode()[0]
    lr_reference_levels.append(most_common_level)

log1p = FunctionTransformer(np.log1p, feature_names_out="one-to-one")

lr_preprocessor = ColumnTransformer(
    [
        ("scale", StandardScaler(), lr_scale_cols),
        ("log_scale", make_pipeline(log1p, StandardScaler()), lr_log_scale_cols),
        ("impute_scale", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), lr_impute_scale_cols),
        ("impute_log_scale", make_pipeline(SimpleImputer(strategy="median"), log1p, StandardScaler()),
         lr_impute_log_scale_cols),
        ("onehot", OneHotEncoder(drop=lr_reference_levels, handle_unknown="ignore", sparse_output=False), LR_ONEHOT_COLS),
        ("binary", "passthrough", lr_passthrough_cols),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

### Check the LR Preprocessor

Fit on the training months only, then apply the same fitted transforms to validation. The checks confirm there are no missing values left, both sets have the same columns, and the scaled training features have mean ~0 and std ~1.

In [10]:
X_train_lr = lr_preprocessor.fit_transform(lr_train_input)
X_val_lr = lr_preprocessor.transform(lr_val_input)

# Logistic regression can't handle NaN, so every missing value must have been imputed
assert X_train_lr.notna().all().all(), "NaNs left in the training features"
assert X_val_lr.notna().all().all(), "NaNs left in the validation features"
assert list(X_train_lr.columns) == list(X_val_lr.columns), "Training and validation columns differ"

print(X_train_lr.shape)
print(X_val_lr.shape)
X_train_lr.describe().T[["mean", "std", "min", "max"]].round(2)

(903157, 50)
(96843, 50)


,mean,std,min,max
credit_risk_score,-0.00,1.00,-4.26,3.69
name_email_similarity,0.00,1.00,-1.70,1.74
velocity_6h,-0.00,1.00,-2.01,3.59
velocity_24h,0.00,1.00,-2.54,3.23
velocity_4w,0.00,1.00,-2.78,2.50
date_of_birth_distinct_emails_4w,0.00,1.00,-1.91,5.74
customer_age,0.00,1.00,-1.96,4.64
income,-0.00,1.00,-1.57,1.18
days_since_request,-0.00,1.00,-0.31,6.39
zip_count_4w,0.00,1.00,-10.82,2.63


- The 33 kept columns become 50 model inputs: 16 scaled numeric columns, 22 one-hot dummies (with the most common level of each dropped as the reference) and 12 0/1 columns passed through.
- Scaled training features have mean 0 and std 1. Validation is transformed with the training statistics, so its means and stds will drift slightly, which is expected.
- A few scaled features still have long tails (e.g. `zip_count_4w` min -10.8, `prev_address_months_count` max 9.5).

## Save Preprocessed Data

Each model gets its own folder containing preprocessed data: `X_train`, `X_val`, `y_train` and `y_val` -> later notebooks only need to read one folder:

```
../data/preprocessed/
    xgb/  X_train, X_val, y_train, y_val  (.parquet)
    lr/   X_train, X_val, y_train, y_val  (.parquet)
```

Parquet keeps dtypes, including the `category` levels for XGBoost, and the original row index, so rows can still be matched back to `Base.csv` for fairness checks later on e.g. `customer_age`.

In [11]:
datasets = {
    "xgb": {"X_train": X_train_xgb, "X_val": X_val_xgb, "y_train": y_train.to_frame(), "y_val": y_val.to_frame()},
    "lr": {"X_train": X_train_lr, "X_val": X_val_lr, "y_train": y_train.to_frame(), "y_val": y_val.to_frame()},
}

for model, splits in datasets.items():
    out_dir = PREPROCESSED_DIR / model
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, data in splits.items():
        data.to_parquet(out_dir / f"{name}.parquet")
        print(f"{model}/{name}.parquet: {data.shape}")

xgb/X_train.parquet: (903157, 33)
xgb/X_val.parquet: (96843, 33)
xgb/y_train.parquet: (903157, 1)
xgb/y_val.parquet: (96843, 1)
lr/X_train.parquet: (903157, 50)
lr/X_val.parquet: (96843, 50)
lr/y_train.parquet: (903157, 1)
lr/y_val.parquet: (96843, 1)


In [12]:
# Round-trip check: every saved file reloads identically, including dtypes and category levels
for model, splits in datasets.items():
    for name, data in splits.items():
        pd.testing.assert_frame_equal(pd.read_parquet(PREPROCESSED_DIR / model / f"{name}.parquet"), data)
print("All saved files reloaded identically")

All saved files reloaded identically
